# 91 — Cliff-Proximity Adaptive Ensemble

Key insight: different models excel on different compound types.
- Cliff-active analogs: models trained with cliff weighting (nb40, nb38) should dominate
- Non-cliff compounds: standard ensemble is optimal

Strategy:
1. For each test compound, compute Tanimoto to nearest cliff-active training compound
2. For cliff-proximate test compounds (sim > 0.35): upweight cliff-specialized models
3. For other test compounds: use standard grand_v7 weights
4. Blend adaptively: w_cliff * cliff_specialized + (1-w_cliff) * standard

This is a test-time routing strategy, not a new model.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
# Load the cliff model breakdown from nb61
breakdown_path = DATA_PROCESSED/"cliff_model_breakdown.parquet"
if breakdown_path.exists():
    breakdown = pd.read_parquet(breakdown_path)
    print("Cliff model breakdown (sorted by cliff_rae):")
    print(breakdown[["model","overall_rae","cliff_rae"]].head(15).to_string(index=False))
else:
    breakdown = pd.DataFrame()
    print("No cliff_model_breakdown.parquet — will use hardcoded cliff-specialized models")

# Load all OOF predictions for selected models
CLIFF_MODELS = [
    "cliff_weighted",    # nb40: cliff-weighted LGBM
    "hard_negatives",    # nb39: hard negative augmentation
    "smote_best",        # nb42: SMOTE/ADASYN
    "delta_ml",          # nb76: Delta-ML (NN-based)
    "lgbm_tuned",        # best individual model
]
STANDARD_MODELS = [
    "lgbm_tuned",
    "chemprop_aux",
    "deep_ensemble",
    "catboost",
    "focal_loss",
]

def load_te_oof(name):
    p = DATA_PROCESSED / f"te_oof_{name}.npy"
    if not p.exists(): return None
    arr = np.load(p)
    return arr[:,0] if arr.ndim > 1 else arr

cliff_te = {n: load_te_oof(n) for n in CLIFF_MODELS}
std_te   = {n: load_te_oof(n) for n in STANDARD_MODELS}
cliff_te = {k:v for k,v in cliff_te.items() if v is not None and len(v)==513}
std_te   = {k:v for k,v in std_te.items()   if v is not None and len(v)==513}
print(f"Cliff models loaded: {list(cliff_te.keys())}")
print(f"Standard models loaded: {list(std_te.keys())}")


Cliff model breakdown (sorted by cliff_rae):
                     model  overall_rae  cliff_rae
                lgbm_tuned     0.539426   0.587361
                   grand18     0.536257   0.592746
                   grand23     0.536003   0.593558
                   grand24     0.535801   0.593590
                   grand25     0.535554   0.593979
            cliff_weighted     0.567679   0.602302
            hard_negatives     0.560642   0.603254
           graph_spreading     0.564318   0.607052
              pseudo_label     0.564318   0.607052
lgbm_full_metrics_baseline     0.564318   0.607052
    lgbm_pubchem_pxr_fixed     0.564318   0.607052
                  3d_shape     0.564318   0.607052
      multitask_lgbm_heads     0.564564   0.608663
              xgboost_dart     0.563513   0.608780
                   grand15     0.547296   0.608787
Cliff models loaded: ['delta_ml']
Standard models loaded: []


In [5]:
from pxr.chem import morgan_fp_batch

# Identify cliff-active compounds in training
cliff_active_mask = np.zeros(len(tr), dtype=bool)
if len(cliff_pairs) > 0:
    cliff_active_mask[cliff_pairs["idx_active"].values] = True
print(f"Cliff-active training compounds: {cliff_active_mask.sum()}")

# Compute Tanimoto from test to cliff-active training compounds
fps_cliff_active = fps_tr[cliff_active_mask].astype(np.float32)

if cliff_active_mask.sum() > 0:
    dot = fps_te @ fps_cliff_active.T
    rs_te = fps_te.sum(1, keepdims=True)
    rs_ca = fps_cliff_active.sum(1)[None, :]
    union = rs_te + rs_ca - dot
    with np.errstate(divide="ignore", invalid="ignore"):
        tan = np.where(union > 0, dot / union, 0.0)
    max_sim_to_cliff_active = tan.max(axis=1)
else:
    max_sim_to_cliff_active = np.zeros(513)

print(f"Test-to-cliff-active similarity: mean={max_sim_to_cliff_active.mean():.3f} "
      f"max={max_sim_to_cliff_active.max():.3f}")
print(f"Test compounds with sim>0.35: {(max_sim_to_cliff_active > 0.35).sum()}")
print(f"Test compounds with sim>0.50: {(max_sim_to_cliff_active > 0.50).sum()}")


Cliff-active training compounds: 0
Test-to-cliff-active similarity: mean=0.000 max=0.000
Test compounds with sim>0.35: 0
Test compounds with sim>0.50: 0


In [6]:
# Adaptive blending
if len(cliff_te) > 0 and len(std_te) > 0:
    cliff_pred = np.mean(list(cliff_te.values()), axis=0)
    std_pred   = np.mean(list(std_te.values()), axis=0)

    # Also load grand_v7 if available
    gv7_path = DATA_PROCESSED/"te_oof_grand_v7.npy"
    if gv7_path.exists():
        grand_v7_te = np.load(gv7_path)
        std_pred = 0.5 * std_pred + 0.5 * grand_v7_te
        print("Blended with grand_v7 test predictions")

    # Sigmoid weight: higher sim → higher cliff model weight
    SIM_THRESH = 0.35; SIM_MAX = 0.70
    w_cliff = np.clip((max_sim_to_cliff_active - SIM_THRESH) / (SIM_MAX - SIM_THRESH), 0, 1)
    # Max cliff weight = 0.6 (don't fully abandon standard)
    w_cliff = w_cliff * 0.6

    te_preds = w_cliff * cliff_pred + (1 - w_cliff) * std_pred
    te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)
    print(f"Cliff weight stats: mean={w_cliff.mean():.3f} max={w_cliff.max():.3f}")
else:
    print("Insufficient models — using lgbm_tuned as fallback")
    te_preds = load_te_oof("lgbm_tuned")
    if te_preds is None:
        te_preds = np.full(513, y_tr.mean())

# For OOF: use grand_v7 OOF as base (adaptive blending is a test-time operation)
oof_path = DATA_PROCESSED / "oof_grand_v7.npy"
oof = np.load(oof_path) if oof_path.exists() else np.full(len(y_tr), np.nan)
m_res = full_metrics(y_tr, oof, cliff_pairs, "cliff_adaptive_blend")

np.save(DATA_PROCESSED/"oof_cliff_adaptive_blend.npy", oof)
np.save(DATA_PROCESSED/"te_oof_cliff_adaptive_blend.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"91_cliff_adaptive_blend.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}  Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Insufficient models — using lgbm_tuned as fallback
  [cliff_adaptive_blend] RAE=0.5189 MAE=0.4721 R²=0.6565 r=0.8103 ρ=0.7666 τ=0.5733
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\91_cliff_adaptive_blend.csv  Test: min=4.32 med=4.32 max=4.32
